# 🤖 Agent Platform — Hands On: a governed, multi-channel workflow engine

**Problem Type A:** *"Design an AI agent platform for non-technical users to configure workflow
automations across multiple channels."*

This notebook builds and runs every piece of it: channel adapters, the tool registry, staged
rollout, routing and conflict resolution, the guardrail decision engine, and the orchestrator's two
core properties - idempotency and durability. No LLM calls anywhere - every decision is
deterministic, so every cell below is fast and exactly reproducible.

## Learning Objectives
In this notebook, you will learn:
1. **Channel adapters** - normalise raw webhook and Slack payloads into one canonical `Event`
2. **Constrained tool schemas** - reject malformed tool arguments before anything executes
3. **Staged rollout** - promote a workflow draft -> testing -> shadow -> live -> autonomous, role-gated and one stage at a time
4. **Routing and the entity lock** - settle a same-trigger conflict by priority and refuse a second concurrent run on one ticket
5. **Guardrail decisions** - watch five ordered rules allow or deny a destructive step, on `LIVE` and on `AUTONOMOUS` workflows
6. **Durable, idempotent execution** - pause for approval, retry without a double refund, crash and resume from a checkpoint, halt a runaway workflow, and read the trace

## Prerequisites
- `docs/01-theory.md` (the design this notebook implements) - read it first if you have not; `docs/03-src-modules-reference.md` describes each `src/agent_platform` module
- Dependencies installed with `uv sync` in this project's folder
- No keys needed: the notebook makes no LLM or network calls, so this project's `.env` can stay empty

---
## 📦 Step 1: Environment Setup

The setup cell loads this project's `.env` (nothing in it is required), finds the project root, and puts `src/` (the `agent_platform` package) and `scripts/` (the shared demo scenario in `_scenario.py`) on `sys.path`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load .env, find the project root, expose src/ and scripts/
# ============================================================================
import sys
import warnings
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# No keys are needed; loading .env keeps the setup identical to the other course notebooks
load_dotenv(find_dotenv(usecwd=True))

# Walk up from the notebook folder until we find src/agent_platform
ROOT = Path.cwd()
while not (ROOT / "src" / "agent_platform").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))      # the agent_platform package
sys.path.insert(0, str(ROOT / "scripts"))  # _scenario.py: the demo workflows and guardrail policy

print("project root:", ROOT)
print("✅ Environment ready")

---
## 📡 Part 1: Channel adapters: one canonical Event

Wildly different raw payload shapes, normalised once, at the edge, into the one shape everything
downstream ever sees. Nothing past this point branches on which channel an event came from.

In [ ]:
# ============================================================================
# CHANNEL ADAPTERS: two very different payloads -> one canonical Event
# ============================================================================
from agent_platform import channels

webhook_event = channels.from_webhook({
    "type": "ticket.created", "id": "wh_1", "tenant_id": "cascade_robotics",
    "ticket_id": "TCK-5510", "order_id": "ORD-2201", "refund_amount_usd": 35.0,
})
slack_event = channels.from_slack({
    "team_id": "cascade_robotics", "ts": "1699999999.001", "text": "prod checkout is failing!",
    "is_urgent": True, "channel_name": "support-escalations",
})

for label, ev in [("webhook", webhook_event), ("slack", slack_event)]:
    print(f"{label:<10}channel={ev.channel.value:<10}event_type={ev.event_type:<18}"
          f"target={ev.target_entity_id}")

---
## 🧰 Part 2: The tool registry: constrained schemas, not free text

"Prefer constrained tool schemas and enums over free-text arguments; validate every tool argument
before execution" (§3.4). A malformed call is rejected before anything executes.

In [ ]:
# ============================================================================
# TOOL REGISTRY: every tool, whether it is destructive, and its typed schema
# ============================================================================
from agent_platform.tools import REGISTRY, validate_args

for name, tool in REGISTRY.items():
    print(f"{name:<16}destructive={tool.destructive!s:<6}schema={tool.schema}")

In [ ]:
# ============================================================================
# ARGUMENT VALIDATION: wrong type and missing field are rejected before execution
# ============================================================================
tool = REGISTRY["issue_refund"]

# Wrong type - rejected before execution.
d = validate_args(tool, {"order_id": "ORD-1", "amount_usd": "fifty dollars"})
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

# Missing a required field - rejected before execution.
d = validate_args(tool, {"order_id": "ORD-1"})
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

# Valid.
d = validate_args(tool, {"order_id": "ORD-1", "amount_usd": 35.0})
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

---
## 🚀 Part 3: Staged rollout: draft -> testing -> shadow -> live -> autonomous

Promotion is role-gated and order-enforced - the same shape as the stage gates in the FDE delivery
framework project (`04_fde_delivery_framework`, `engine.advance_stage()`). Only an `admin` or
`approver` can promote, so a workflow's author (role `author`) can never promote it themselves.

**Function-by-function explanation of the cell below:**

- `get_principal(user_id)` — looks up a hardcoded demo `Principal` (id, display name, role) by user id; raises `KeyError` if unknown.
- `build_store()` — creates a `WorkflowStore` and publishes three demo `WorkflowSpec`s into it (`wf_ticket_triage`, `wf_legacy_ticket_tagger`, `wf_runaway`), returning the populated store.
- `default_policy()` — returns a demo `GuardrailPolicy` for the tenant with a $50 spend cap, a max of 10 steps, and only `tag_ticket` on the autonomous allow-list (so `issue_refund` always needs a human approver).
- `promote(store, workflow_id, to_status, signer)` — advances a workflow's latest version to the next rollout stage; denies if the workflow is unknown, if `signer` isn't `admin`/`approver` (so an author can't self-promote), or if `to_status` would skip a stage — otherwise mutates the spec's status and returns an "allowed" `Decision`.

Everything else in the cell (`store`, `policy`, `author`, `wrong_hat`, `approver`, `admin`) are just the return values of the calls above, and `WorkflowStatus` is the enum ordering the rollout stages (`DRAFT → TESTING → SHADOW → LIVE → AUTONOMOUS`).

In [ ]:
# ============================================================================
# STAGED ROLLOUT: wrong role and skipped stage are denied; one stage at a time passes
# ============================================================================
from _scenario import build_store, default_policy, TENANT

from agent_platform.identity import get_principal
from agent_platform.models import WorkflowStatus
from agent_platform.workflows import promote

store = build_store()
policy = default_policy()
author = get_principal("u_author_dana")
wrong_hat = get_principal("u_author_wrong_hat")
approver = get_principal("u_approver_raj")
admin = get_principal("u_admin_lee")

# An author-role user cannot promote the workflow (Dana wrote it).
d = promote(store, "wf_ticket_triage", WorkflowStatus.TESTING, wrong_hat)
print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

# Cannot skip a stage.
d = promote(store, "wf_ticket_triage", WorkflowStatus.AUTONOMOUS, admin)
print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

# The correct path, one stage at a time.
for target in [WorkflowStatus.TESTING, WorkflowStatus.SHADOW, WorkflowStatus.LIVE]:
    d = promote(store, "wf_ticket_triage", target, admin)
    print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

---
## 🔀 Part 4: Routing: priority and the entity lock

Two independent mechanisms. Priority is a *design-time* choice - which workflow should win a
same-trigger conflict. The entity lock is a *run-time* safety property - can two workflows ever be
mutating the same ticket at once - and it holds even if priority was misconfigured.

**Function-by-function explanation of the cell below:**

- `channels.from_webhook(payload)` — normalises the raw webhook payload into a canonical `Event` (same adapter used in Part 1); here it builds a second `ticket.created` event for `TCK-5510`.
- `store.all_live(TENANT)` — returns the latest-version `WorkflowSpec` for every workflow this tenant has. Despite the name it does not filter by status (`route()` drops drafts); here all three are `LIVE`: `wf_ticket_triage` (promoted in Part 3), `wf_legacy_ticket_tagger` (published `LIVE` from the start) and `wf_runaway` (triggers on `ticket.updated`, so it never matches this event).
- `routing.route(event, live_workflows)` — picks exactly one workflow for the event, or explains why none ran:
  - internally calls `matching_workflows()`, which filters to workflows for the same tenant, not in `DRAFT`, with a trigger matching the event's `channel` + `event_type`;
  - denies `no_trigger_match` if nothing matches, or `entity_locked` if the event's target entity already has an active run;
  - otherwise picks the **highest-priority** matching trigger as the winner and returns an "allowed" `Decision` naming it.

Since both `wf_ticket_triage` (priority `10`) and `wf_legacy_ticket_tagger` (priority `1`) match the same `webhook` / `ticket.created` trigger, `route()` deterministically picks `wf_ticket_triage` — priority breaks the tie, not registration order or any randomness.

In [ ]:
# ============================================================================
# LIVE CHECK: both workflows that match the ticket.created trigger are LIVE now
# ============================================================================
status_wf_legacy_ticket_tagger = store.latest("wf_legacy_ticket_tagger")
status_wf_ticket_triage = store.latest("wf_ticket_triage")
print(status_wf_legacy_ticket_tagger.status)
print(status_wf_ticket_triage.status)

In [ ]:
# ============================================================================
# ROUTING: two LIVE workflows match one event - priority picks the winner
# ============================================================================
from agent_platform import routing

event = channels.from_webhook({
    "type": "ticket.created", "id": "wh_2", "tenant_id": TENANT,
    "ticket_id": "TCK-5510", "order_id": "ORD-2201", "refund_amount_usd": 35.0,
})
live_workflows = store.all_live(TENANT)
d = routing.route(event, live_workflows)
print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")
print("(the high-priority ticket-triage workflow wins over the low-priority legacy tagger,")
print(" even though both match the same trigger)")

In [ ]:
# ============================================================================
# ENTITY LOCK: a second concurrent run on the same ticket is refused
# ============================================================================
# The exclusivity lock is independent of priority.
assert routing.acquire_lock("TCK-locked-demo", "run_a") is True
locked_again = routing.acquire_lock("TCK-locked-demo", "run_b")
print(f"second concurrent acquire on the same ticket: {locked_again}")
routing.release_lock("TCK-locked-demo")
print(f"after release, acquire succeeds again: {routing.acquire_lock('TCK-locked-demo', 'run_c')}")
routing.release_lock("TCK-locked-demo")

---
## 🛡️ Part 5: The guardrail decision

Five ordered rules, deny overrides - the same shape as `authz.policy.decide()` in the RAG project
and `gates.py::sign_off()` in the delivery framework.

### 5.1 🚦 The refund step on the `LIVE` workflow

`probe()` runs the workflow's refund step through `authorize_step()` without executing it: once with no approver, once with Raj (role `approver`), and once with a cost over the $50 tenant spend cap.

In [ ]:
# ============================================================================
# GUARDRAIL PROBE: the LIVE refund step without approval, with approval, over the cap
# ============================================================================
from agent_platform import orchestrator
from agent_platform.guardrails import authorize_step
from agent_platform.models import Run

refund_tool = REGISTRY["issue_refund"]

def probe(workflow, cost=0.0, approval=None, **run_kwargs):
    run = Run(run_id="probe", workflow_id=workflow.workflow_id, workflow_version=1, event=event)
    step = workflow.steps[1]   # the refund step
    return authorize_step(workflow, run, step, refund_tool, cost, policy, approval=approval)

live_wf = store.get_version("wf_ticket_triage", 1)
d = probe(live_wf)
print(f"LIVE, no approver          -> {'ALLOW' if d.allowed else 'DENY'} [{d.rule}]")
d = probe(live_wf, approval=approver)
print(f"LIVE, with approver        -> {'ALLOW' if d.allowed else 'DENY'} [{d.rule}]")
d = probe(live_wf, cost=999.0, approval=approver)
print(f"LIVE, over the spend cap   -> {'ALLOW' if d.allowed else 'DENY'} [{d.rule}]")

### 5.2 🔓 Autonomous raises the ceiling, it does not remove it

`AUTONOMOUS` status skips human approval only for destructive tools the tenant policy allow-lists, and `default_policy()` does not allow-list `issue_refund`. The cell below builds a separate store, promotes its copy of `wf_ticket_triage` all the way to `AUTONOMOUS`, and probes the same refund step: it still needs a human, and the spend cap still applies. The main `store` is untouched, so Parts 6-8 keep running against the `LIVE` workflow.

In [ ]:
# ============================================================================
# AUTONOMOUS PROBE: the same refund step on a fully autonomous workflow
# ============================================================================
auto_store = build_store()   # a fresh, separate store - Parts 6-8 keep using `store`
for target in [WorkflowStatus.TESTING, WorkflowStatus.SHADOW, WorkflowStatus.LIVE, WorkflowStatus.AUTONOMOUS]:
    d = promote(auto_store, "wf_ticket_triage", target, admin)
    print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

auto_wf = auto_store.get_version("wf_ticket_triage", 1)
print(f"\nissue_refund allow-listed for autonomous runs: {'issue_refund' in policy.allowed_destructive_tools_autonomous}")
d = probe(auto_wf)
print(f"AUTONOMOUS, no approver    -> {'ALLOW' if d.allowed else 'DENY'} [{d.rule}]")
d = probe(auto_wf, cost=999.0, approval=approver)
print(f"AUTONOMOUS, over the cap   -> {'ALLOW' if d.allowed else 'DENY'} [{d.rule}]")

---
## ⏸️ Part 6: The full run: a real destructive step, paused, then approved

`run_workflow()` executes the `draft` step, then stops at the refund: the workflow is `LIVE` and no approver was passed, so the run pauses instead of failing. `resume()` continues from the checkpoint with Raj's approval.

In [ ]:
# ============================================================================
# FULL RUN: pause at the destructive step, then resume with an approver
# ============================================================================
run = orchestrator.run_workflow("nb_run_001", live_wf, event, policy)
print(f"state after first pass: {run.state.value}")

run = orchestrator.resume(run, live_wf, policy, approval=approver)
print(f"state after approval:   {run.state.value}  total_cost=${run.total_cost_usd:.2f}")

---
## 🔁 Part 7: Idempotency: a retry never double-applies a side effect

The cell rewinds the checkpoint so the refund step is attempted again, the way a redelivered event or a worker retry would. The idempotency key (`run_id:step_name`) already exists, so the retry is a no-op: the external refund counter does not move.

In [ ]:
# ============================================================================
# IDEMPOTENCY: retry the refund step and count the real external calls
# ============================================================================
before = orchestrator.external_call_count("issue_refund")
run.next_step_index = 1   # rewind the checkpoint to re-attempt the refund step only
run = orchestrator.resume(run, live_wf, policy, approval=approver)
after = orchestrator.external_call_count("issue_refund")
print(f"issue_refund real calls before retry: {before}")
print(f"issue_refund real calls after retry:  {after}")
print(f"the retried step's side_effect_applied flag: {run.completed_steps[-1].side_effect_applied}")

---
## 💾 Part 8: Durability: a simulated crash, then resume from the checkpoint

`crash_after_step=0` stops the run right after the `draft` step, as a process restart would. `resume()` reads `next_step_index` (the checkpoint) and continues from the refund step, so `draft` runs exactly once.

In [ ]:
# ============================================================================
# DURABILITY: crash after step 0, then resume from the checkpoint
# ============================================================================
event2 = channels.from_webhook({
    "type": "ticket.created", "id": "wh_3", "tenant_id": TENANT,
    "ticket_id": "TCK-5599", "order_id": "ORD-2299", "refund_amount_usd": 8.0,
})
run2 = orchestrator.run_workflow("nb_run_002", live_wf, event2, policy, crash_after_step=0)
print(f"after simulated crash: state={run2.state.value}  next_step_index={run2.next_step_index}")

run2 = orchestrator.resume(run2, live_wf, policy, approval=approver)
draft_executions = [e for e in run2.events if e["kind"] == "step_executed" and e["step"] == "draft"]
print(f"state after resume: {run2.state.value}")
print(f"'draft' step executed {len(draft_executions)} time(s) total - never re-ran after the crash")

---
## 🛑 Part 9: Step budget: a runaway workflow halts with a named reason

`wf_runaway` is a misconfigured workflow with 30 `tag_ticket` steps and its own budget of `max_steps=5`. The guardrail uses the tighter of the tenant policy (10 steps) and the workflow's budget, so the run stops after 5 steps with state `halted_budget` and a named rule, instead of looping until someone notices.

In [ ]:
# ============================================================================
# STEP BUDGET: run the misconfigured 30-step workflow against a fresh ticket
# ============================================================================
runaway = store.get_version("wf_runaway", 1)
event3 = channels.from_webhook({
    "type": "ticket.updated", "id": "wh_4", "tenant_id": TENANT, "ticket_id": "TCK-5601",
})
run3 = orchestrator.run_workflow("nb_run_003", runaway, event3, policy)
print(f"state={run3.state.value}  steps completed={len(run3.completed_steps)} of {len(runaway.steps)} declared")

# The last step_authorize event names the rule that stopped the run
denied = [e for e in run3.events if e["kind"] == "step_authorize" and not e["allowed"]]
print(f"halting rule: [{denied[-1]['rule']}] {denied[-1]['reason']}")

---
## 🔍 Part 10: Observability: the full, replayable trace

Every authorization, execution and crash in Part 8 was logged on `run2.events` as it happened; `render_run()` only formats that record.

In [ ]:
# ============================================================================
# TRACE: render every logged event of the crashed-then-resumed run
# ============================================================================
from agent_platform.observability import render_run

print(render_run(run2))

---
## 📝 Summary

In this notebook, we learned:

### 1. Channel adapters and constrained tool schemas
- Every channel is normalised once, at the edge, into one `Event`; nothing downstream branches on the channel.
- **The model decides THAT something should happen; a typed, validated schema decides HOW.**
  Constrained arguments, checked before execution, are where determinism actually lives.

### 2. Staged rollout and the guardrail decision
- Promotion is role-gated and one stage at a time; an author cannot promote their own workflow.
- **Autonomous status raises the ceiling; it never removes it.** A non-allow-listed destructive
  tool still needs a human even on a fully autonomous workflow, and the spend cap applies at every
  stage.

### 3. Routing and the entity lock
- **Priority and the entity lock answer two different questions.** Which workflow should win a
  conflict, versus whether two can ever run concurrently - independent checks, on purpose.

### 4. Durable, idempotent execution
- **One execution loop, not two.** `resume()` and a fresh run share the exact same code path -
  there's no separate "recovery" path with its own bugs.
- **An idempotency key belongs to the action, not the run.** A run can legitimately retry; one
  specific side effect must never apply twice.
- **A halted run with a named reason beats a workflow that loops until someone notices.**

### Next Steps
- **`../INTERVIEW_SCRIPT.md`**: how to present all of this on a whiteboard in 60 minutes (this is the project's only notebook).
- **`scripts/demo_guardrail_failure.py`**: the same negative controls as one script (`uv run python scripts/demo_guardrail_failure.py` from the project folder).